In [ ]:
import pandas as pd
import json
from datetime import datetime
from typing import List, Dict, Any
import numpy as np
import os
import sys

raw_data_dir = "/content/data/raw/"
output_dir = "/content/data/fingerprints/"

def parse_scan_results(data: Dict[str, Any]) -> pd.DataFrame:
    rows = []
    ble_data = data.get('bleData', {})
    for device_id, scans in ble_data.items():
        for scan in scans:
            scan_copy = scan.copy()
            scan_copy['device_id'] = device_id
            if 'serviceData' in scan_copy:
                try:
                    scan_copy['serviceData'] = json.dumps(scan_copy['serviceData'])
                except TypeError:
                    scan_copy['serviceData'] = str(scan_copy['serviceData'])
            rows.append(scan_copy)
    if not rows: return pd.DataFrame(columns=['rssi', 'timestamp', 'device_id'])
    df = pd.DataFrame(rows)
    if 'rssi' not in df.columns: df['rssi'] = np.nan
    if 'timestamp' not in df.columns: df['timestamp'] = pd.NaT
    if 'device_id' not in df.columns: df['device_id'] = None
    return df

# Funzione create_radiomap (usa MEDIANA)
def create_radiomap(df: pd.DataFrame, window_seconds: int = 3, window_step: int = 1, duration_seconds: int = None, default_rssi: float = 110.0) -> (pd.DataFrame, pd.Timestamp, int):
    if df.empty or 'timestamp' not in df.columns or df['timestamp'].isnull().all():
        return pd.DataFrame(), None, 0
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp').dropna(subset=['timestamp'])
    df.replace(0, np.nan, inplace=True)
    if df.empty: return pd.DataFrame(), None, 0

    start_time = df['timestamp'].min()
    if duration_seconds is not None:
        num_windows = max(0, int((duration_seconds - window_seconds) / window_step) + 1)
    else:
        end_time = df['timestamp'].max()
        actual_duration = (end_time - start_time).total_seconds()
        num_windows = max(0, int((actual_duration - window_seconds) / window_step) + 1)

    windows = []
    all_device_ids = df['device_id'].unique()

    for window_idx in range(num_windows):
        window_start = start_time + pd.Timedelta(seconds=window_idx * window_step)
        window_end = window_start + pd.Timedelta(seconds=window_seconds)
        window_data = df[(df['timestamp'] >= window_start) & (df['timestamp'] < window_end)]
        mean_row = {'window_idx': window_idx}
        for device_id in all_device_ids: mean_row[device_id] = default_rssi
        if not window_data.empty:
            rssi_medians = window_data.groupby('device_id')['rssi'].median().reset_index()
            for _, row in rssi_medians.iterrows():
                 if pd.notna(row['rssi']):
                     mean_row[row['device_id']] = row['rssi']
        windows.append(mean_row)

    radiomap = pd.DataFrame(windows)
    return radiomap, start_time, num_windows

# NUOVA FUNZIONE PER SENSORI
def create_sensor_windows(sensor_data: List[Dict[str, Any]], sensor_name: str, start_time: pd.Timestamp, num_windows: int, window_seconds: int = 3, window_step: int = 1) -> pd.DataFrame:
    """Crea finestre di statistiche per UN tipo di sensore."""
    if not sensor_data: return pd.DataFrame(index=range(num_windows))

    df = pd.DataFrame(sensor_data)
    if 'timestamp' not in df.columns or df['timestamp'].isnull().all():
        return pd.DataFrame(index=range(num_windows))
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp').dropna(subset=['timestamp'])
    if df.empty: return pd.DataFrame(index=range(num_windows))

    sensor_axes = ['x', 'y', 'z']
    for col in sensor_axes:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        else:
             df[col] = np.nan

    windows_stats = []
    for window_idx in range(num_windows):
        window_start_ts = start_time + pd.Timedelta(seconds=window_idx * window_step)
        window_end_ts = window_start_ts + pd.Timedelta(seconds=window_seconds)
        window_data = df[(df['timestamp'] >= window_start_ts) & (df['timestamp'] < window_end_ts)]

        stats = {'window_idx': window_idx}
        for axis in sensor_axes:
            data_axis = window_data[axis].dropna()
            if len(data_axis) > 1:
                stats[f'{sensor_name}_{axis}_mean'] = data_axis.mean()
                stats[f'{sensor_name}_{axis}_std'] = data_axis.std()
            elif len(data_axis) == 1:
                stats[f'{sensor_name}_{axis}_mean'] = data_axis.iloc[0]
                stats[f'{sensor_name}_{axis}_std'] = 0.0
            else:
                stats[f'{sensor_name}_{axis}_mean'] = np.nan
                stats[f'{sensor_name}_{axis}_std'] = np.nan
        windows_stats.append(stats)

    return pd.DataFrame(windows_stats)
# Funzione process_json_data MODIFICATA per Sensor Fusion
def process_json_data(docs: List[Dict[str, Any]], min_rssi_length: int = 100, window_seconds: int = 3, window_step: int = 1, sensors_to_process: List[str] = None, default_rssi: float = 110.0) -> pd.DataFrame:
    try:
        all_data = []
        all_beacon_columns = set()
        sensors_to_process = sensors_to_process or ['accelerometerData', 'gyroscopeData', 'magnetometerData']

        processed_docs = []
        for doc in docs:
            artwork = doc.get("artwork")
            room = doc.get("room")
            platform = doc.get("platform", "unknown")
            duration = doc.get("recordingDurationSeconds")
            rssi_df = parse_scan_results(doc)

            if not rssi_df.empty and rssi_df.shape[0] > min_rssi_length:
                # Crea Radiomap (mediana) e ottieni info finestre
                radiomap, doc_start_time, num_windows_in_doc = create_radiomap(rssi_df, window_seconds, window_step, duration, default_rssi)

                if not radiomap.empty and num_windows_in_doc > 0:
                    current_beacons = set(col for col in radiomap.columns if col != 'window_idx')
                    all_beacon_columns.update(current_beacons)

                    # Crea Finestre Sensori
                    sensor_dfs = [radiomap] # Inizia con radiomap
                    for sensor_name in sensors_to_process:
                        sensor_data = doc.get(sensor_name, [])
                        sensor_window_df = create_sensor_windows(sensor_data, sensor_name.replace('Data',''), doc_start_time, num_windows_in_doc, window_seconds, window_step)
                        
                        if 'window_idx' in sensor_window_df.columns and sensor_window_df.index.name == 'window_idx':
                             sensor_window_df = sensor_window_df.reset_index(drop=True)
                        elif 'window_idx' not in sensor_window_df.columns:
                             sensor_window_df['window_idx'] = range(num_windows_in_doc)

                        if not sensor_window_df.empty:
                            sensor_dfs.append(sensor_window_df)

                    # Mergia Radiomap e Finestre Sensori
                    if len(sensor_dfs) > 1:
                         combined_df = sensor_dfs[0]
                         for df_to_merge in sensor_dfs[1:]:
                              # Controllo robusto per il merge
                              if 'window_idx' in combined_df.columns and 'window_idx' in df_to_merge.columns:
                                   combined_df = pd.merge(combined_df, df_to_merge, on="window_idx", how="left")
                              else:
                                   print(f"Attenzione: Impossibile fare il merge per mancanza di 'window_idx'. Sensore: {sensor_name}")
                    else:
                         combined_df = radiomap # Solo radiomap se non ci sono sensori validi

                    combined_df['artwork'] = artwork
                    combined_df['room'] = room
                    combined_df['platform'] = platform
                    processed_docs.append(combined_df)

        if not processed_docs: return pd.DataFrame()

        # Allinea colonne e concatena (simile a prima)
        final_beacon_columns = sorted(list(all_beacon_columns), key=lambda x: (int(x.split('-')[0]), int(x.split('-')[1])) if '-' in x else (999, 999))
        all_sensor_stats_columns = sorted([f'{s.replace("Data","")}_{ax}_{stat}' for s in sensors_to_process for ax in ['x','y','z'] for stat in ['mean', 'std']])

        aligned_docs = []
        for df in processed_docs:
            for col in final_beacon_columns:
                if col not in df.columns: df[col] = default_rssi
            for col in all_sensor_stats_columns:
                 if col not in df.columns: df[col] = np.nan # NaN per stats sensori mancanti
            aligned_docs.append(df)

        final_df = pd.concat(aligned_docs, ignore_index=True)

        # Gestione NaN: Prima riempi i beacon, poi i sensori (es. con 0 o media)
        final_df[final_beacon_columns] = final_df[final_beacon_columns].fillna(default_rssi)
        final_df[all_sensor_stats_columns] = final_df[all_sensor_stats_columns].fillna(0.0)


        # Rimuovi colonne non necessarie
        final_df.drop(columns=['window_idx', 'platform'], inplace=True, errors='ignore')

        # Riordina le colonne: [beacons] + [sensor_stats] + [room, artwork]
        room_artwork_cols = ['room', 'artwork']
        final_ordered_cols = final_beacon_columns + all_sensor_stats_columns + room_artwork_cols
        final_ordered_cols = [col for col in final_ordered_cols if col in final_df.columns]
        final_df = final_df[final_ordered_cols]

        return final_df
    except Exception as e:
        print(f"Errore durante il processamento dei dati JSON con sensori: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame()

# Funzione load_session_data
def load_session_data(file_path, min_rssi_length: int = 100, window_seconds: int = 3, window_step: int = 1, sensors_to_process: List[str] = None, default_rssi: float = 110.0):
    print(f"Caricamento e processamento (con sensori) di: {file_path}")
    if not os.path.exists(file_path):
        print(f"Errore: File non trovato - {file_path}")
        return pd.DataFrame()
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            content = file.read()
            try:
                 data = json.loads(content)
                 if not isinstance(data, list):
                      data = [data]
            except json.JSONDecodeError:
                 print(f"Formato JSON non standard rilevato in {file_path}. Tentativo di lettura riga per riga.")
                 data = [json.loads(line) for line in content.strip().split('\n') if line.strip()]

        df = process_json_data(data, min_rssi_length, window_seconds, window_step, sensors_to_process, default_rssi)
        return df
    except json.JSONDecodeError as e:
        print(f"Errore grave nel decodificare JSON da {file_path}: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"Errore inaspettato durante il caricamento di {file_path}: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame()

# Funzione main MODIFICATA per Sensor Fusion
def main():
    print("Avvio della creazione dei fingerprint arricchiti con SENSOR FUSION (basati su MEDIANA)...")
    os.makedirs(output_dir, exist_ok=True)

    raw_files = [
        "android_session_1_1.json",
        "android_session_2_2.json",
        "ios_session_1_3.json",
        "ios_session_2_4.json"
    ]
    # Sensori da includere nel fingerprint
    sensors = ['accelerometerData', 'gyroscopeData', 'magnetometerData']

    for raw_file in raw_files:
        raw_file_path = os.path.join(raw_data_dir, raw_file)
        print("-" * 30)

        base_name = raw_file.replace('.json', '')
        output_file_name = f"{base_name}_fusion_median.csv"
        output_file_path = os.path.join(output_dir, output_file_name)
        # Carica e processa i dati includendo i sensori
        df = load_session_data(raw_file_path, sensors_to_process=sensors)

        if not df.empty:
            df.to_csv(output_file_path, index=False)
            print(f"Salvataggio completato: {output_file_path} ({df.shape[0]} righe, {df.shape[1]} colonne)")
            print("Prime colonne:", list(df.columns[:5]) + ['...'] + list(df.columns[-5:]))
        else:
            print(f"Nessun dato valido generato per {raw_file}, file CSV non creato.")

    print("-" * 30)
    print("Processamento Sensor Fusion completato con successo!")

# Esecuzione diretta
if __name__ == "__main__":
    main()